# 01 — Value Iteration en GridWorld

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1PB2XFEsQV4GXcg1pad7r3ml1FIIhUbOr)

## Notación

- Estado: $s=(row,col)$
- Acción: $a=(\Delta row,\Delta col)$
- Transición:
  $$
  T(s,a,s')=P(s'\mid s,a)
  $$
- Recompensa:
  $$
  R(s)
  $$
- Bellman de optimalidad:
  $$
  V_{k+1}(s)=R(s)+\gamma\max_a
  \sum_{s'}T(s,a,s')V_k(s')
  $$

La recompensa está escrita como **$R(s)$**, tal como en el notebook de la clase.



## Puente teoría ↔ código

La ecuación que implementaremos es:

$$
V_{k+1}(s)=
R(s)+
\gamma
\max_a
\sum_{s'}
T(s,a,s')V_k(s')
$$

En el código, cada término aparece así:

| Teoría | Código |
|---|---|
| $s$ | `state` |
| $a$ | `action` |
| $s'$ | `next_state` |
| $R(s)$ | `grid.get_reward(state)` |
| $\gamma$ | `grid.gamma` |
| $T(s,a,s')$ | `prob` dentro de `get_transition_probs(...)` |
| $\sum_{s'}$ | `sum(...)` |
| $\max_a$ | `max(... for action in grid.actions)` |
| $V_k(s')$ | `V[next_state]` |
| $V_{k+1}(s)$ | `V_new[state]` |

La idea es leer el código exactamente como la ecuación de Bellman.


## 1. Construcción del MDP

Usamos una clase `GridWorld` para que toda la definición del MDP esté encapsulada en un solo objeto.

In [1]:
import numpy as np

class GridWorld:
    """
    GridWorld usando la notación:
      state  = (row, col)
      action = (dr, dc)
      R(s)   = recompensa del estado actual
      T(s,a,s') = P(s' | s,a)
    """

    def __init__(self, height=3, width=4):
        self.height = height
        self.width = width

        # Estados especiales
        self.wall = (1, 1)
        self.terminal_states = {
            (0, 3): +1.0,
            (1, 3): -1.0,
        }

        # Parámetros del MDP
        self.living_reward = -0.04
        self.gamma = 1.0
        self.p_intended = 0.8
        self.p_perpendicular = 0.1

        # Right, Down, Left, Up
        self.actions = [
            (0, 1),
            (1, 0),
            (0, -1),
            (-1, 0),
        ]

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state != self.wall

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # Un estado terminal es absorbente:
        # una vez allí, el agente permanece en el mismo estado.
        if self.is_terminal(state):
            return self.terminal_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        # Esta función representa la dinámica del MDP:
        #
        #     T(s,a,s') = P(s' | s,a)
        #
        # Recibe un estado s y una acción a, y devuelve los
        # estados siguientes posibles s' con sus probabilidades.
        """
        Devuelve [(next_state, probability), ...]
        para T(s,a,s') = P(s' | s,a).
        """
        # Un estado terminal es absorbente:
        # una vez allí, el agente permanece en el mismo estado.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Perpendiculares a (dr,dc)
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        # Construimos los posibles resultados de ejecutar la acción.
        # En un mundo estocástico, la acción intentada puede desviarse.
        outcomes = [
            (action, self.p_intended),
            (perp1, self.p_perpendicular),
            (perp2, self.p_perpendicular),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )

            # Si el movimiento sale del grid o choca con una pared,
            # el agente no se mueve: s' = s.
            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions

In [2]:
grid = GridWorld()

print("Estados:")
print(grid.states())

print("\nEjemplo de transición:")
s = (2, 0)
a = (-1, 0)  # UP
print("s =", s, "a =", a)
print(grid.get_transition_probs(s, a))
print("Suma =", sum(p for _, p in grid.get_transition_probs(s, a)))

Estados:
[(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3)]

Ejemplo de transición:
s = (2, 0) a = (-1, 0)
[((1, 0), 0.8), ((2, 0), 0.1), ((2, 1), 0.1)]
Suma = 1.0


## 2. Value Iteration

Para una acción fija definimos:

$$
Q_V(s,a)=\sum_{s'}T(s,a,s')V(s')
$$

y la actualización es:

$$
V_{k+1}(s)=R(s)+\gamma\max_a Q_{V_k}(s,a)
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # Implementa la parte estocástica de Bellman:
    #
    #     Σ_{s'} T(s,a,s') V(s')
    #
    # Una acción no tiene por qué llevar a un único estado siguiente.
    # get_transition_probs(state, action) devuelve todos los posibles
    # s' junto con sus probabilidades T(s,a,s').
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-6, max_iter=10_000):
    # Inicialización de Value Iteration:
    # V_0(s) = 0 para todos los estados.
    # Antes de iterar, todavía no hemos propagado ninguna recompensa.
    V = {state: 0.0 for state in grid.states()}
    deltas = []


    # Cada vuelta de este ciclo corresponde a una iteración k.
    # En cada iteración calcularemos V_{k+1} a partir de V_k.
    for iteration in range(max_iter):
        # Usamos una copia para hacer una actualización sincrónica:
        # todos los V_{k+1}(s) se calculan usando únicamente V_k.
        V_new = V.copy()
        # Este valor implementa el criterio de convergencia:
        # max_s |V_{k+1}(s) - V_k(s)|.
        biggest_change = 0.0


        # Aplicamos la ecuación de Bellman a cada estado s.
        for state in grid.states():
            # En un estado terminal ya no hay decisiones futuras.
            # Su valor queda fijado por su recompensa R(s).
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:

                # Esta línea corresponde a:
                #
                #   max_a Σ_{s'} T(s,a,s') V_k(s')
                #
                # Para cada acción a:
                # 1. se calculan los posibles estados siguientes s',
                # 2. se pondera V_k(s') por T(s,a,s'),
                # 3. se suman esos valores esperados,
                # 4. se elige la acción con mayor valor esperado.
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )


                # Ecuación de Bellman de optimalidad:
                #
                # V_{k+1}(s) = R(s)
                #                + gamma * max_a Σ_{s'} T(s,a,s') V_k(s')
                #
                # En el código:
                # grid.get_reward(state)  -> R(s)
                # grid.gamma              -> gamma
                # best_expected_value     -> max_a Σ T V_k
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
            # Cambio local del estado:
            # |V_{k+1}(s) - V_k(s)|
                abs(V_new[state] - V[state])
            )


        # Terminamos la iteración:
        # V_{k+1} pasa a ser V_k para la siguiente vuelta.
        V = V_new
        deltas.append(biggest_change)


        # Si ningún estado cambia más que theta, consideramos
        # que Value Iteration ha convergido.
        if biggest_change < threshold:
            break

    return V, iteration + 1, deltas

In [4]:
V_star, iterations, deltas = value_iteration(grid)

print(f"Convergió en {iterations} iteraciones")
print("\nV*(s):")
ARROWS = {
    (0, 1): "→",
    (1, 0): "↓",
    (0, -1): "←",
    (-1, 0): "↑",
}

def print_values(grid, V, fmt="{:+.3f}"):
    for row in range(grid.height):
        line = []
        for col in range(grid.width):
            s = (row, col)
            if not grid.is_valid_state(s):
                line.append("  WALL  ")
            else:
                line.append(fmt.format(V[s]))
        print(" | ".join(line))

def print_policy(grid, policy):
    for row in range(grid.height):
        line = []
        for col in range(grid.width):
            s = (row, col)
            if not grid.is_valid_state(s):
                line.append(" # ")
            elif grid.is_terminal(s):
                line.append(" + " if grid.get_reward(s) > 0 else " - ")
            else:
                line.append(f" {ARROWS[policy[s]]} ")
        print(" | ".join(line))

print_values(grid, V_star)

Convergió en 30 iteraciones

V*(s):
+0.812 | +0.868 | +0.918 | +1.000
+0.762 |   WALL   | +0.660 | -1.000
+0.705 | +0.655 | +0.611 | +0.388


## 3. Extraer la política óptima

Una vez tenemos $V^*$:

$$
\pi^*(s)=
\arg\max_a
\sum_{s'}T(s,a,s')V^*(s')
$$

Observa que aquí tampoco necesitamos añadir nuevamente $R(s)$: para un estado fijo es igual para todas las acciones.


In [5]:
def extract_policy(grid, V):
    # Una vez tenemos V*(s), extraemos la política óptima.
    # La política guarda una acción por cada estado no terminal.
    policy = {}


    # Evaluamos cada estado de manera independiente.
    for state in grid.states():
        if grid.is_terminal(state):
            continue


        # Policy extraction:
        #
        # pi*(s) = argmax_a Σ_{s'} T(s,a,s') V*(s')
        #
        # Aquí max(..., key=...) no devuelve el valor máximo:
        # devuelve la ACCIÓN que produce ese máximo.
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(
                grid, state, action, V
            )
        )

    return policy


policy_star = extract_policy(grid, V_star)

print("π*(s):")
print_policy(grid, policy_star)

π*(s):
 →  |  →  |  →  |  + 
 ↑  |  #  |  ↑  |  - 
 ↑  |  ←  |  ←  |  ← 


## 4. Idea para recordar

**Value Iteration mezcla evaluación y mejora en una sola actualización.**

El `max` aparece dentro de Bellman porque en cada estado estamos buscando directamente la mejor acción.